<a href="https://colab.research.google.com/github/nihemelandu/customer-lifetime-value/blob/main/%5Cnotebooks%5CChurn_prediction_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
def load_customer_data(file_path='customer_analytics_master.csv'):
    """
    Load customer analytics data from CSV file.
    Returns pandas DataFrame with basic info printed.
    """
    print("🔄 Loading customer analytics data...")
    try:
        # Load the CSV file
        df = pd.read_csv(file_path)

        # Display basic information
        print("✅ Data loaded successfully!")
        print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
        print(f"🏷️  Columns: {list(df.columns)}")

        # Convert order_date to datetime
        df['order_date'] = pd.to_datetime(df['order_date'])

        # Show data types
        print("\n📋 Data types:")
        print(df.dtypes)

        # Show first few rows
        print("\n👀 First 3 rows:")
        print(df.head(3))

        # Basic stats
        print("\n📈 Basic info:")
        print(f"   • Unique customers: {df['customer_id'].nunique():,}")
        print(f"   • Unique orders: {df['order_id'].nunique():,}")
        print(f"   • Date range: {df['order_date'].min()} to {df['order_date'].max()}")

    except FileNotFoundError:
        print(f"❌ Error: File not found at {file_path}")
        return None
    except Exception as e:
        print(f"❌ Error loading data: {str(e)}")
        return None

    return df


In [33]:
import pandas as pd
import numpy as np

data = load_customer_data()

🔄 Loading customer analytics data...
✅ Data loaded successfully!
📊 Shape: 200,367 rows × 23 columns
🏷️  Columns: ['customer_id', 'order_id', 'registration_date', 'customer_segment', 'acquisition_channel', 'account_status', 'order_date', 'order_value', 'order_status', 'quantity', 'unit_price', 'category', 'price_tier', 'base_price', 'effective_date', 'price', 'pricing_strategy', 'end_date', 'promotion_name', 'promotion_type', 'discount_percentage', 'start_date', 'end_date_1']

📋 Data types:
customer_id                     int64
order_id                       object
registration_date              object
customer_segment               object
acquisition_channel            object
account_status                 object
order_date             datetime64[ns]
order_value                   float64
order_status                   object
quantity                      float64
unit_price                    float64
category                       object
price_tier                     object
base_price 

In [11]:
from churn_pipeline import ChurnPredictionPipeline


In [15]:
churnpredict = ChurnPredictionPipeline()
df = churnpredict.load_customer_data(file_path='customer_analytics_master.csv')

🔄 Loading customer analytics data...
✅ Data loaded successfully!
📊 Shape: 200,367 rows × 23 columns
🏷️  Columns: ['customer_id', 'order_id', 'registration_date', 'customer_segment', 'acquisition_channel', 'account_status', 'order_date', 'order_value', 'order_status', 'quantity', 'unit_price', 'category', 'price_tier', 'base_price', 'effective_date', 'price', 'pricing_strategy', 'end_date', 'promotion_name', 'promotion_type', 'discount_percentage', 'start_date', 'end_date_1']

📋 Data types:
customer_id                     int64
order_id                       object
registration_date              object
customer_segment               object
acquisition_channel            object
account_status                 object
order_date             datetime64[ns]
order_value                   float64
order_status                   object
quantity                      float64
unit_price                    float64
category                       object
price_tier                     object
base_price 

In [16]:
df.account_status.unique()

array(['active', 'inactive', 'suspended', 'closed'], dtype=object)

In [26]:
df = df.loc[~df.order_date.isna()]

In [27]:
df.loc[df.account_status== 'inactive',].order_date.max(), df.order_date.max()

(Timestamp('2024-07-30 00:00:00'), Timestamp('2024-07-30 00:00:00'))

In [29]:
df.loc[df.account_status== 'inactive',].head()

,customer_id,order_id,registration_date,customer_segment,acquisition_channel,account_status,order_date,order_value,order_status,quantity,...,base_price,effective_date,price,pricing_strategy,end_date,promotion_name,promotion_type,discount_percentage,start_date,end_date_1
579,17368,ord_55e0be0f,2022-10-25 00:00:00,high_value,direct,inactive,2022-11-19,104.12,delivered,1.0,...,442.05,2021-06-05,442.05,regular,NaN,Up-sized didactic product 2022,flash_sale,0.221,2022-11-12,2022-12-12
580,17368,ord_55e0be0f,2022-10-25 00:00:00,high_value,direct,inactive,2022-11-19,104.12,delivered,1.0,...,442.05,2021-09-05,349.37,clearance,NaN,Up-sized didactic product 2022,flash_sale,0.221,2022-11-12,2022-12-12
581,17368,ord_8606b5e1,2022-10-25 00:00:00,high_value,direct,inactive,2024-05-23,147.16,delivered,1.0,...,104.15,2021-09-30,104.15,regular,NaN,NaN,NaN,NaN,NaN,NaN
582,17368,ord_8606b5e1,2022-10-25 00:00:00,high_value,direct,inactive,2024-05-23,147.16,delivered,1.0,...,104.15,2022-02-12,107.07,competitive_match,NaN,NaN,NaN,NaN,NaN,NaN
583,17368,ord_8606b5e1,2022-10-25 00:00:00,high_value,direct,inactive,2024-05-23,147.16,delivered,1.0,...,104.15,2022-04-16,67.34,clearance,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
print(f"Dataset Shape: {df.shape}")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Dataset Shape: (189798, 23)
Memory Usage: 148.17 MB


In [31]:
print("\nMissing Values:")
missing_summary = df.isnull().sum()
missing_pct = (missing_summary / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_summary,
    'Missing_Percentage': missing_pct
}).sort_values('Missing_Count', ascending=False)
print(missing_df[missing_df.Missing_Count > 0])


Missing Values:
                     Missing_Count  Missing_Percentage
end_date                    189798           100.00000
start_date                  161767            85.23114
end_date_1                  161767            85.23114
discount_percentage         161767            85.23114
promotion_name              161767            85.23114
promotion_type              161767            85.23114


In [34]:
print("\nBasic Statistics for Numeric Columns:")
numeric_cols = df.select_dtypes(include=[np.number]).columns
print(df[numeric_cols].describe())


Basic Statistics for Numeric Columns:
         customer_id    order_value       quantity     unit_price  \
count  189798.000000  189798.000000  189798.000000  189798.000000   
mean    22432.172710      79.132152       1.252252      96.798539   
std      7202.555087      51.720684       0.538579     115.923620   
min     10000.000000      10.000000       1.000000       9.580000   
25%     16227.000000      38.390000       1.000000      32.260000   
50%     22398.000000      66.160000       1.000000      53.490000   
75%     28660.000000     103.290000       1.000000     110.460000   
max     34997.000000     341.000000       3.000000     786.360000   

          base_price          price  end_date  discount_percentage  
count  189798.000000  189798.000000       0.0          28031.00000  
mean       96.825785      88.249137       NaN              0.26548  
std       115.969211     108.468207       NaN              0.13781  
min        10.030000       3.020000       NaN              0.05

In [35]:
# ==========================================
# 2. TARGET VARIABLE CREATION & ANALYSIS
# ==========================================
print("\n\n2. TARGET VARIABLE DEEP DIVE")
print("-" * 40)

# Convert date columns
date_cols = ['registration_date', 'order_date', 'effective_date', 'start_date', 'end_date', 'end_date_1']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')




2. TARGET VARIABLE DEEP DIVE
----------------------------------------
